# Web문서를 읽어 RAG구현
- Web에서 Text 추출 -> Embedding -> ChromaDB에 저장 -> 질문과 유사한 문서를 검색 -> LLM에게 요청 후 결과 생성

In [1]:
!pip install openai sentence-transformers chromadb python-dotenv
!pip install requests beautifulsoup4 # python 버전 (전통적인 방법)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 95.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.1 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found 

In [4]:
from openai import OpenAI
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
from chromadb import PersistentClient
import os
from bs4 import BeautifulSoup
import requests
from dotenv import load_dotenv

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
model = "gpt-4o-mini"
embedder = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [17]:
# web page에서 텍스트 읽기
def extract_from_urlFunc(url):
  headers = {'User-Agent':"Mozila/5.0"} # chrome 엔진

  resp = requests.get(url, headers=headers)
  # print("status_code :", resp.status_code) # 성공 : 200

  if resp.status_code != 200:
    print("요청 실패, html preview", resp.text[:200])
    return []

  soup = BeautifulSoup(resp.text, "html.parser")
  paragraphs = soup.find_all("p")

  texts = [ # p tag안의 텍스트만 빼내고 빈 문자열 제외
      p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True) # 문자열 양쪽 공백 제거
  ]
  print("found p count :",len(texts))

  return texts

# Chroma DB Collection 설정
chroma_client = PersistentClient(path="./chroma_db")

# Colab은 반복 실행할 수 있기 때문에 DB초기화가 필요
try:
  chroma_client.delete_collection("webdata")
except:
  pass

collection = chroma_client.get_or_create_collection(
        name="webdata",
        metadata={"hnsw:space": "cosine"},
)

# url 읽어오기
url = "https://ko.wikipedia.org/wiki/%EA%B9%80%EC%B9%98%EC%B0%8C%EA%B0%9C"
web_docs = extract_from_urlFunc(url)
print(web_docs)

if not web_docs:
  print("추출된 문서 없음")
  raise Exception("문서 없음")

# web_docs를 벡터로 변환
web_embeddings = embedder.encode(web_docs, normalize_embeddings=True)
print("web_embeddings :", web_embeddings.shape) # (9, 384)

# Chroma DB 저장
for i, (doc, emb) in enumerate(zip(web_docs, web_embeddings), start=0):
  collection.add(
      documents=[doc],
      embeddings=[emb.tolist()],
      ids=[f'web_{i}'],
  )
print('저장된 문서 수 : ', collection.count())

result = collection.get(ids=["web_0","web_1"], include=['documents', 'embeddings'])
print("샘플 문서 :", result['documents'])
# print("샘플 임베딩 :", result['embeddings'])

found p count : 9
['국물류', '반찬', '김치찌개는 대표적인한국 요리가운데 하나로,김치를 넣고 얼큰하게 끓인찌개이다.[1]된장찌개·순두부찌개와 함께 가장 널리 알려진 요리이다.', '김치찌개는고추,배추,소금,고춧가루등을 이용한김치가 등장한 이후 생겨난 것으로 보고 있다. 김치가 지나치게 시어 염도가 높아 그대로 먹기 어려워지거나, 양을 늘리기 위해 물을 넣어 끓여 먹던 방식에서 비롯된 것으로 여겨진다.궁중 요리에도 소개되었으며, 궁중에서는 ‘김치조치’라고 불렸다고 한다.', '재료가 비교적 간단하고 조리하기 쉬워한국의 가정에서 흔히 만들어 먹는다. 김치찌개에는 배추김치와채소,두부,육류,어패류등이 들어가며, 보통 육류와 해산물을 함께 넣지는 않는다. 육류로는 주로돼지고기를 사용하며, 쇠고기는 질겨지기 쉬워 일반적으로는 잘 쓰지 않는다. 해산물로는참치를 많이 넣고,고등어나꽁치를 사용하기도 한다. 다른 한국 요리와 마찬가지로밥과 함께 곁들여 먹는 경우가 많다. 끓이는 과정에서 고춧가루를 더하면 맛과 색이 한층 진해진다.', '김치찌개에는 어느 정도발효되어 신맛이 나는 김치를 사용하는데, 담근 지 얼마 되지 않은 김치를 쓰면 특유의 깊은 맛이 덜하다. 신 김치를 그대로 사용하기도 하지만, 기름에 볶은 뒤 끓이면 맛이 더 깊어진다. 볶을 때는참기름이나들기름을 주로 사용하며, 돼지고기를 넣을 경우 라드(돼지 비계)로 볶기도 한다.', '고기나 해산물을 넣을 때 잡내를 줄이기 위해된장을 약간 풀어 넣기도 한다. 된장 대신청주나소주등을 넣기도 하며, 조리 중간에 넣는마늘과양파도 잡내 제거에 도움이 된다. 매우 신 김치를 사용할 경우에는 별도의 재료를 더하지 않아도 잡내가 줄어든다. 신맛이 강할 때에는고추장,설탕,소금등을 더해 맛을 조절하기도 한다.', '영양 면에서는 김치 외에 다양한 재료를 더해 영양소를 보충할 수 있다. 다만 김치는 높은 온도에서 끓이면유산균이 감소한다. 또한 김치와 부재료로 인해나트륨함량이 높은 편이며, 볶는 과정에서 사용하는 기름 때

In [23]:
# LLM에게 질문(prompt)하기 위한 질문 내용 강화 - Augmented
query = "김치찌개의 역사와 조리법 알려 줘"

query_vec = embedder.encode([query], normalize_embeddings=True)
print("query_vec :", query_vec.shape) # (1, 384)
# print("query_vec :", query_vec)

# 질문 Vector와 유사한 문서 검색
results = collection.query(
    query_embeddings=query_vec.tolist(),
    n_results=2,
    include=["documents", "distances", "metadatas"],
)
retrieved_docs = results['documents'][0]
retrieved_dist = results['distances'][0]

for i, (doc, dist) in enumerate(zip(retrieved_docs, retrieved_dist), 1):
  print(f'문서{i} : {doc}')
  print(f'유사도{i} : {dist:.4f}\n')

# 검색 기반 프롬프트 구성
retrieved_text = '\n'.join(retrieved_docs)
print(retrieved_text)

query_vec : (1, 384)
문서1 : 김치찌개는 대표적인한국 요리가운데 하나로,김치를 넣고 얼큰하게 끓인찌개이다.[1]된장찌개·순두부찌개와 함께 가장 널리 알려진 요리이다.
유사도1 : 0.2854

문서2 : 영양 면에서는 김치 외에 다양한 재료를 더해 영양소를 보충할 수 있다. 다만 김치는 높은 온도에서 끓이면유산균이 감소한다. 또한 김치와 부재료로 인해나트륨함량이 높은 편이며, 볶는 과정에서 사용하는 기름 때문에지방함량도 비교적 높다. 이에 따라칼로리도 높은 편에 속한다.
유사도2 : 0.4231

김치찌개는 대표적인한국 요리가운데 하나로,김치를 넣고 얼큰하게 끓인찌개이다.[1]된장찌개·순두부찌개와 함께 가장 널리 알려진 요리이다.
영양 면에서는 김치 외에 다양한 재료를 더해 영양소를 보충할 수 있다. 다만 김치는 높은 온도에서 끓이면유산균이 감소한다. 또한 김치와 부재료로 인해나트륨함량이 높은 편이며, 볶는 과정에서 사용하는 기름 때문에지방함량도 비교적 높다. 이에 따라칼로리도 높은 편에 속한다.


In [26]:
# 프롬프트 생성 - Generation
prompt = f"""
너는 한국요리 특히 찌개 전문가야
다음 문서를 참고하여 '{query}'에 대해 답변해줘

[참고 문서]
{retrieved_text}

[요구 사항]
김치찌개의 조리법, 재료의 역할까지 설명해
최소 20문장 이내로 작성해
마크다운 없이 평문으로 작성해
각 문장을 줄바꿈으로 분리해
"""

# print('prompt 내용 : ', prompt)

# LLM에게 질문 후 결과 얻기
response = client.responses.create(
    model=model,
    input=prompt
)
print(f"[답변] \n{response.output_text}") # 질문이 하나인데도 오래걸려 -> 이제는 비동기화가 필요

[답변] 
김치찌개는 한국을 대표하는 전통 음식 중 하나로, 매콤하고 얼큰한 맛이 특징입니다.

이 찌개는 주로 잘 익은 김치를 활용하여 조리됩니다.

김치는 발효된 배추로, 유산균이 풍부해 건강에 많은 이점을 제공합니다.

그러나 높은 온도에서 끓일 경우 유산균이 감소할 수 있으니 주의가 필요합니다.

김치찌개는 보통 여러 가지 재료와 함께 조리해 그 영양가를 높입니다.

주로 사용되는 추가 재료로는 돼지고기, 두부, 양파, 대파, 마늘 등이 있습니다.

돼지고기는 찌개의 풍미를 더해주며, 기름기 있는 부위가 특히 추천됩니다.

두부는 찌개의 부드러운 식감을 주고, 단백질 보충에도 도움이 됩니다.

양파와 대파는 단맛과 향을 더해주어 깊은 풍미를 부여합니다.

마늘은 향신료 역할을 하여 건강에도 좋고, 맛을 한층 끌어올려 줍니다.

찌개를 끓일 때는 먼저 고기를 볶아 기름과 맛을 내는 것이 중요합니다.

이후에 김치를 추가하고 함께 볶아주면 김치의 맛이 더 진해집니다.

다음으로 물을 추가한 후, 다른 재료들을 넣고 끓입니다.

조리 과정에서 간단한 양념으로 고춧가루, 국간장, 소금을 추가해 맛을 조절합니다.

기호에 따라 매운 정도를 조절할 수 있어 개인 취향에 맞게 만들 수 있습니다.

김치찌개는 보통 밥과 함께 뚝배기에 담아 서빙합니다.

직접 뜨거운 상태에서 먹으면 더욱 맛있습니다.

이 찌개는 특히 추운 날씨에 인기가 많은데, 따뜻한 국물이 몸을 데워주기 때문입니다.

김치찌개는 다양한 변형이 있으며, 각 가정마다 고유의 레시피가 존재합니다.

정통 방식부터 현대적 변형까지, 김치찌개는 한국 음식의 다양성을 잘 보여줍니다.

이처럼 김치찌개는 역사와 전통을 담고 있는 음식으로, 지금도 많은 사랑을 받고 있습니다.
